In [ ]:
'''
                   User
                     │
                     ▼
              Orchestrator Agent
                     │
      ┌──────────────┼───────────────┐
      ▼              ▼               ▼
 Idea Generator   Market Analyst   Trend Analyst
      │              │               │
      ▼              ▼               ▼
 Product Designer ── Debate Loop ── Finance Agent
      │                                   │
      ▼                                   ▼
 Tech Architect                    Marketing Agent
      │                                   │
      ▼                                   ▼
 Code Generator                     Website Generator
      │                                   │
      ▼                                   ▼
             Pitch Deck Generator
                     │
                     ▼
                 Final Report


## Agent stracture

               USER
                 │
                 ▼
         Startup Builder Crew
                 │
 ┌────────────────────────────────┐
 │                                │
 ▼                                ▼

Trend Agent                 Market Agent
     │                          │
     ▼                          ▼
 Idea Generator             VC Critic
           │
           ▼
      Product Designer
           │
           ▼
      Tech Architect
           │
           ▼
      Code Generator
           │
           ▼
      GitHub Repo Agent
           │
           ▼
      Website Generator
           │
           ▼
      Deployment Agent
           │
           ▼
       Pitch Deck Agent
           │
           ▼
        Final Report


                 
def startup_builder(topic):

    idea = idea_agent.run(topic)
    
    trend = trend_agent.run(idea)
    
    market = market_agent.run(idea)

    product = product_agent.run(idea, market, trend)

    finance = finance_agent.run(product)

    tech = tech_architect_agent.run(product)

    marketing = marketing_agent.run(product)

    website = website_agent.run(product)

    pitch = pitch_agent.run(product, finance)

    return {
        "idea": idea,
        "market": market,
        "product": product,
        "finance": finance,
        "tech": tech,
        "marketing": marketing,
        "website": website,
        "pitch": pitch
    }
    

                 
autonomous_vc_agent/
│
├── agents/
│
│   ├── idea_agent.py ## 2,3
│   ├── trend_agent.py
│   ├── market_agent.py ## 1
│   ├── product_agent.py ## 4
│   ├── tech_architect_agent.py
│   ├── finance_agent.py
│   ├── marketing_agent.py ## 5
│   ├── website_agent.py
│   └── pitchdeck_agent.py
│
├── tools/
│
│   ├── web_search.py
│   ├── trend_analysis.py
│   └── competitor_analysis.py
│
├── workflows/
│
│   └── startup_graph.py
│
├── memory/
│   └── vector_store.py
│
├── reports/
│
├── app.py
├── config.py
└── requirements.txt




Trend Agent
   │
   └── Tavily + Serper

Market Agent
   │
   └── Tavily + Serper + Web Scraper

Idea Agent
   │
   └── Tavily

Marketing Agent
   │
   └── Tavily + Serper

Finance Agent
   │
   └── Python Tool

Tech Architect
   │
   └── Tavily

Code Generator
   │
   └── Python Tool

Final Report
   │
   └── File Writer
'''

In [ ]:
# ! pip install langchain_experimental==0.4.1

In [1]:
import os
from langchain_tavily import TavilySearch
from langchain_community.tools import DuckDuckGoSearchRun
from crewai_tools import FileWriterTool
from langchain_community.tools import RequestsGetTool
from langchain_community.tools import RequestsPostTool
from langchain_experimental.tools.python.tool import PythonREPLTool

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7, api_key=os.getenv("OPENAI_API_KEY"))

# -----------------------------
# Tools
# -----------------------------

tavily_tool = TavilySearch(api_key=os.getenv("TAVILY_API_KEY"))

# search = DuckDuckGoSearchRun() ## pip install -U ddgs

python_tool = PythonREPLTool()

file_writer = FileWriterTool()

# web_scraper = RequestsGetTool()

/home/ahmed/miniconda3/envs/ai/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ! pip install 'crewai[tools]' tavily-python

In [ ]:
import os
from typing import List,Dict
from pydantic import BaseModel, ConfigDict, Field

# Core CrewAI imports
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool 

# Specialized Tool imports
from crewai_tools import FileWriterTool, TavilySearchTool
from langchain_experimental.utilities import PythonREPL

# =========================================================
# 1. Tools Initialization
# =========================================================
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

# Pre-built tools
tavily_tool = TavilySearchTool()
file_writer = FileWriterTool()

# Custom Wrapped Python Tool (Fixed the shadowing issue)
@tool("python_repl")
def python_repl_tool(command: str) -> str:
    """Execute python code in a REPL. Useful for calculating financial models 
    or processing data. Input should be valid python code."""
    return PythonREPL().run(command)


# =========================================================
# 3. Agents
# =========================================================

# Ensure llm is defined elsewhere in your notebook/script
# llm = ChatOpenAI(model="gpt-4o") 

trend_agent = Agent(
    role="Trend Analyst",
    goal="Identify emerging technology and startup trends.",
    backstory="Expert in spotting early market signals.",
    tools=[tavily_tool],
    verbose=True,
    llm=llm
)

market_agent = Agent(
    role="Market Research Analyst",
    goal="Analyze market opportunities.",
    backstory="Former strategy consultant specializing in emerging tech markets.",
    tools=[tavily_tool],
    verbose=True,
    llm=llm
)

idea_agent = Agent(
    role="Startup Strategist",
    goal="Generate innovative startup ideas.",
    backstory="Venture capitalist with deep experience in SaaS startups.",
    tools=[tavily_tool],
    verbose=True,
    memory=True,
    llm=llm
)

critic_agent = Agent(
    role="VC Critic",
    goal="Evaluate startup ideas and pick the best one.",
    backstory="Experienced venture capitalist evaluating investment opportunities.",
    tools=[tavily_tool],
    verbose=True,
    llm=llm
)

product_agent = Agent(
    role="Product Designer",
    goal="Design MVP product specifications.",
    backstory="Product manager experienced in launching SaaS products.",
    tools=[tavily_tool],
    verbose=True,
    llm=llm
)

marketing_agent = Agent(
    role="Marketing Strategist",
    goal="Create a go-to-market strategy.",
    backstory="Growth marketer specializing in startup launches.",
    tools=[tavily_tool],
    verbose=True,
    llm=llm
)

finance_agent = Agent(
    role="Financial Analyst",
    goal="Create financial projections and funding plans.",
    backstory="Investment banker experienced in SaaS financial modeling.",
    tools=[python_repl_tool],
    verbose=True,
    llm=llm
)

tech_architect_agent = Agent(
    role="Tech Architect",
    goal="Design scalable infrastructure.",
    backstory="Cloud architect with experience deploying large-scale systems.",
    tools=[tavily_tool],
    verbose=True,
    llm=llm
)

code_agent = Agent(
    role="Code Generator",
    goal="Generate MVP code structure.",
    backstory="Full-stack developer capable of building SaaS systems.",
    tools=[python_repl_tool],
    verbose=True,
    llm=llm
)

website_agent = Agent(
    role="Website Generator",
    goal="Generate landing page layout and code.",
    backstory="UX/UI designer specialized in startup landing pages.",
    tools=[tavily_tool],
    verbose=True,
    llm=llm
)

pitch_agent = Agent(
    role="Pitch Deck Generator",
    goal="Create a startup investor pitch deck.",
    backstory="Startup advisor experienced in fundraising.",
    tools=[tavily_tool],
    verbose=True,
    llm=llm
)

report_agent = Agent(
    role="Final Report Generator",
    goal="Compile the full startup blueprint.",
    backstory="Project manager compiling investor-ready reports.",
    tools=[file_writer],
    verbose=True,
    llm=llm
)

# =========================================================
# 4. Tasks
# =========================================================

trend_task = Task(
    description="Identify top 5 emerging AI and sustainability trends.",
    expected_output="List of trends with confidence scores.",
    agent=trend_agent

)
market_task = Task(
    description="Identify high-growth markets and competitor gaps.",
    expected_output="Detailed market research.",
    agent=market_agent,
    context=[trend_task],
)
idea_task = Task(
    description="Generate 3 startup ideas based on the market analysis.",
    expected_output="Detailed startup ideas.",
    agent=idea_agent,
    context=[market_task]
)
critic_task = Task(
    description="Evaluate the generated startup ideas.",
    expected_output="Evaluation report and winning idea.",
    agent=critic_agent,
    context=[idea_task]
)

design_task = Task(
    description="Design an MVP for the winning startup idea.",
    expected_output="Product specification document.",
    agent=product_agent,
    context=[critic_task]
)

marketing_task = Task(
    description="Create a marketing strategy.",
    expected_output="12-month marketing plan.",
    agent=marketing_agent,
    context=[design_task]
)

finance_task = Task(
    description="Generate financial projections.",
    expected_output="Startup financial model.",
    agent=finance_agent,
    context=[marketing_task]
)

architecture_task = Task(
    description="Design technical system architecture.",
    expected_output="Technical architecture document.",
    agent=tech_architect_agent,
    context=[design_task, finance_task]
)

code_task = Task(
    description="Generate MVP code structure.",
    expected_output="Repository structure.",
    agent=code_agent,
    context=[architecture_task]
)

website_task = Task(
    description="Generate landing page HTML and Tailwind CSS.",
    expected_output="Landing page code.",
    agent=website_agent,
    context=[marketing_task],
    output_file="index.html"
)

pitch_task = Task(
    description="Create a 10-slide investor pitch deck.",
    expected_output="Pitch deck outline.",
    agent=pitch_agent,
    context=[marketing_task, finance_task]
)

final_task = Task(
    description="Compile all outputs into a complete startup blueprint.",
    expected_output="Investor-ready startup report.",
    agent=report_agent,
    output_file='research/startup_blueprint.md',
    context=[
        trend_task, market_task, idea_task, critic_task, design_task,
        marketing_task, finance_task, architecture_task, code_task
    ]
)

# =========================================================
# 5. Crew Execution
# =========================================================

startup_crew = Crew(
    agents=[
        trend_agent, market_agent, idea_agent, critic_agent,
        product_agent, marketing_agent, finance_agent,
        tech_architect_agent, code_agent, website_agent,
        pitch_agent, report_agent
    ],
    tasks=[
        trend_task, market_task, idea_task, critic_task,
        design_task, marketing_task, finance_task,
        architecture_task, code_task, website_task,
        pitch_task, final_task
    ],
    process=Process.sequential,
    verbose=True
)

# if __name__ == "__main__":
#     result = startup_crew.kickoff()
#     print("\n==============================")
#     print("FINAL STARTUP BLUEPRINT GENERATED")
#     print("==============================\n")

In [29]:
# =========================================================
# Run System
# =========================================================

if __name__ == "__main__":
    result = startup_crew.kickoff()

    print("\n==============================")
    print("FINAL STARTUP BLUEPRINT")
    print("==============================\n")

    print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: ea4daef9-47c7-4248-9a0c-9039999f8778                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Identify top 5 emerging AI and sustainability trends.                                                    │
│  ID: 74e4d9f9-a18e-4162-b502-626aeb1688d2                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Trend Analyst                                                                                           │
│                                                                                                                 │
│  Task: Identify top 5 emerging AI and sustainability trends.                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: tavily_search                                                                                            │
│  Args: {'query': 'sustainability trends 2024'}                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: tavily_search                                                                                            │
│  Args: {'query': 'emerging AI trends 2024'}                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Final Report Generator                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Investor-Ready Startup Report: AI-Powered Virtual Nursing Assistant (AVNA)                                   │
│                                                                                                                 │
│  ## Table of Contents                                                                                           │
│  1. Executive Summary                                                                                           │
│  2. Market Analysis                                                                                             │
│  3. Startup Ideas                                                                                               │
│  4. Product Specification Document                                                                              │
│  5. 12-Month Marketing Plan                                                                                     │
│  6. Financial Projections                                                                                       │
│  7. Technical Architecture                                                                                      │
│  8. Project Directory Structure                                                                                 │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 1. Executive Summary                                                                                        │
│  The AI-Powered Virtual Nursing Assistant (AVNA) aims to revolutionize patient care by leveraging artificial    │
│  intelligence to enhance monitoring, engagement, and compliance in healthcare. This report outlines the         │
│  startup's market opportunity, product specifications, marketing strategies, financial projections, technical   │
│  architecture, and project structure.                                                                           │
│                                                                                                                 │
│  ## 2. Market Analysis                                                                                          │
│  ### 2.1 Top 5 Emerging AI Trends (2024)                                                                        │
│  1. **Generative AI Scaling**                                                                                   │
│     - Description: Generative AI is expected to reshape industries with significant ROI, particularly in        │
│  financial services, media, and mobility.                                                                       │
│     - Confidence Score: 99.9%                                                                                   │
│     - Source: [Decimal Point                                                                                    │
│  Analytics](https://decimalpointanalytics.com/who-we-are/newsroom/navigating-the-ai-landscape-key-trends-shapi  │
│  ng-2025-and-beyond)                                                                                            │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Compile all outputs into a complete startup blueprint.                                                   │
│  Agent: Final Report Generator                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL STARTUP BLUEPRINT

# Investor-Ready Startup Report: AI-Powered Virtual Nursing Assistant (AVNA)

## Table of Contents
1. Executive Summary  
2. Market Analysis  
3. Startup Ideas  
4. Product Specification Document  
5. 12-Month Marketing Plan  
6. Financial Projections  
7. Technical Architecture  
8. Project Directory Structure  

---

## 1. Executive Summary
The AI-Powered Virtual Nursing Assistant (AVNA) aims to revolutionize patient care by leveraging artificial intelligence to enhance monitoring, engagement, and compliance in healthcare. This report outlines the startup's market opportunity, product specifications, marketing strategies, financial projections, technical architecture, and project structure.

## 2. Market Analysis
### 2.1 Top 5 Emerging AI Trends (2024)
1. **Generative AI Scaling**  
   - Description: Generative AI is expected to reshape industries with significant ROI, particularly in financial services, media, and mobility.  
   - Confidence Score: 99.9%  


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: ea4daef9-47c7-4248-9a0c-9039999f8778                                                                       │
│  Final Output: # Investor-Ready Startup Report: AI-Powered Virtual Nursing Assistant (AVNA)                     │
│                                                                                                                 │
│  ## Table of Contents                                                                                           │
│  1. Executive Summary                                                                                           │
│  2. Market Analysis                                                                                             │
│  3. Startup Ideas                                                                                               │
│  4. Product Specification Document                                                                              │
│  5. 12-Month Marketing Plan                                                                                     │
│  6. Financial Projections                                                                                       │
│  7. Technical Architecture                                                                                      │
│  8. Project Directory Structure                                                                                 │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 1. Executive Summary                                                                                        │
│  The AI-Powered Virtual Nursing Assistant (AVNA) aims to revolutionize patient care by leveraging artificial    │
│  intelligence to enhance monitoring, engagement, and compliance in healthcare. This report outlines the         │
│  startup's market opportunity, product specifications, marketing strategies, financial projections, technical   │
│  architecture, and project structure.                                                                           │
│                                                                                                                 │
│  ## 2. Market Analysis                                                                                          │
│  ### 2.1 Top 5 Emerging AI Trends (2024)                                                                        │
│  1. **Generative AI Scaling**                                                                                   │
│     - Description: Generative AI is expected to reshape industries with significant ROI, particularly in        │
│  financial services, media, and mobility.                                                                       │
│     - Confidence Score: 99.9%                                                                                   │
│     - Source: [Decimal Point                                                                                    │
│  Analytics](https://decimalpointanalytics.com/who-we-are/newsroom/navigating-the-ai-landscape-key-trends-shapi  │
│  ng-2025-and-beyond)                                                                                            │
│                                                       

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
"""
Professional Autonomous Startup Builder (Enterprise Edition)
===========================================================

Author: Ahmed & Gemini
Version: 2.0.0

A modular multi-agent system using CrewAI to research, plan, and architect 
a startup from scratch. Features modular crew execution and structured 
Pydantic validation.
"""

import os
from typing import List, Optional
from pydantic import BaseModel, Field

from crewai import Agent, Task, Crew, Process
from langchain_tavily import TavilySearch
from crewai_tools import PythonREPLTool, FileWriterTool, FileReadTool

# =========================================================
# 1. ENHANCED DATA SCHEMAS
# =========================================================

class StartupIdea(BaseModel):
    name: str = Field(..., description="Catchy name of the startup")
    problem: str = Field(..., description="The specific pain point being solved")
    solution: str = Field(..., description="The unique value proposition")
    revenue_model: str = Field(..., description="How the company makes money")

class IdeaOutput(BaseModel):
    ideas: List[StartupIdea]
    selected_winner_index: int

class TechArchitecture(BaseModel):
    frontend: str
    backend: str
    database: str
    infrastructure: str = Field(..., description="Cloud provider and services")
    deployment_pipeline: str

# =========================================================
# 2. TOOL INITIALIZATION
# =========================================================

tavily_tool = TavilySearch(api_key=os.getenv("TAVILY_API_KEY"))
python_tool = PythonREPLTool()
file_writer = FileWriterTool()
file_reader = FileReadTool()

# =========================================================
# 3. AGENT DEFINITIONS (ROLE-BASED)
# =========================================================

def create_startup_agents():
    """Initializes specialized agents with strict guardrails."""
    
    # RESEARCHER: High temperature for creativity
    researcher = Agent(
        role="Market Intelligence Lead",
        goal="Identify high-alpha technology trends and market gaps.",
        backstory="Ex-Gartner analyst with a knack for spotting 'The Next Big Thing'.",
        tools=[tavily_tool],
        verbose=True,
        allow_delegation=False
    )

    # STRATEGIST: Balanced logic
    strategist = Agent(
        role="Startup Architect",
        goal="Synthesize research into viable, VC-backable startup models.",
        backstory="Serial entrepreneur who has exited three SaaS companies.",
        tools=[tavily_tool],
        verbose=True,
        memory=True
    )

    # ENGINEER: Low temperature for precision
    engineer = Agent(
        role="CTO / Tech Architect",
        goal="Design high-scale, cost-effective technical infrastructure.",
        backstory="Former Staff Engineer at AWS/Google specialized in distributed systems.",
        tools=[python_tool],
        verbose=True
    )

    return researcher, strategist, engineer

# =========================================================
# 4. TASK ORCHESTRATION
# =========================================================

def run_startup_builder():
    """Executes the startup building workflow in phases."""
    
    researcher, strategist, engineer = create_startup_agents()

    # PHASE 1: STRATEGY & IDEATION
    # -----------------------------------------------------
    research_task = Task(
        description="Analyze 2026 trends in AI Agents and Sustainable Energy.",
        expected_output="A list of 5 validated market opportunities with signal strength.",
        agent=researcher
    )

    ideation_task = Task(
        description="Generate 3 startup ideas based on the research provided.",
        expected_output="Three distinct startup profiles with problem/solution fit.",
        agent=strategist,
        context=[research_task],
        output_pydantic=IdeaOutput
    )

    strategy_crew = Crew(
        agents=[researcher, strategist],
        tasks=[research_task, ideation_task],
        process=Process.sequential,
        verbose=True
    )

    print("\n[PHASE 1] Starting Market Research & Ideation...")
    strategy_result = strategy_crew.kickoff()
    
    # HUMAN-IN-THE-LOOP CHECKPOINT (Conceptual for this script)
    # In a production UI, you would pause here for user selection.
    
    # PHASE 2: TECHNICAL ARCHITECTURE
    # -----------------------------------------------------
    arch_task = Task(
        description="Design the full-stack architecture for the winning idea.",
        expected_output="Detailed tech stack, DB schema, and cloud infrastructure plan.",
        agent=engineer,
        output_pydantic=TechArchitecture
    )

    plan_task = Task(
        description="Create a README.md and a project roadmap based on the tech stack.",
        expected_output="A professional markdown project blueprint.",
        agent=engineer,
        tools=[file_writer],
        context=[arch_task]
    )

    engineering_crew = Crew(
        agents=[engineer],
        tasks=[arch_task, plan_task],
        process=Process.sequential,
        verbose=True
    )

    print("\n[PHASE 2] Starting Technical Blueprinting...")
    final_blueprint = engineering_crew.kickoff()

    return {
        "strategy": strategy_result,
        "technical": final_blueprint
    }

# =========================================================
# 5. EXECUTION
# =========================================================

if __name__ == "__main__":
    try:
        results = run_startup_builder()
        print("\n" + "="*50)
        print("BUILDER COMPLETE: Startup assets generated.")
        print("="*50)
    except Exception as e:
        print(f"Workflow failed at runtime: {e}")